In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from the_well.data import WellDataset
from torch.utils.data import DataLoader

from modules import *
from datasets import *

In [ ]:
CHECKPOINT = "checkpoints/best_model.pt"
DATASET_PATH = "/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase"
N_EXAMPLES = 5
BATCH_SIZE = 1

In [ ]:
test_dataset = WellDataset(path=DATASET_PATH, well_split_name="test")
test_ds = HelmholtzDataset(test_dataset)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
x, y = test_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

ckpt = torch.load(CHECKPOINT, map_location="cuda")

model = FNO2d(
    modes1=16,
    modes2=16,
    width=32,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=4
).cuda()
model = torch.compile(model)
model.load_state_dict(ckpt["model_state"])
model.eval()

;

In [ ]:
all_rel_l2 = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.cuda(non_blocking=True)
        y = y.cuda(non_blocking=True)

        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(x)

        rel = (
            torch.norm(pred - y, dim=(-3, -2)) /
            torch.norm(y,        dim=(-3, -2))
        ).mean().item()

        all_rel_l2.append(rel)

In [ ]:
mean_rel_l2 = np.mean(all_rel_l2)
# "acurácia": fração de amostras com erro relativo < 10%
accuracy = np.mean(np.array(all_rel_l2) < 0.1)

print(f"Relative L2 médio : {mean_rel_l2:.6f}")
print(f"Acurácia (< 0.10) : {accuracy * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(
    N_EXAMPLES,
    3,
    figsize=(14, 4 * N_EXAMPLES),
    constrained_layout=True
)

with torch.no_grad():
    for i in range(N_EXAMPLES):

        x, y = test_ds[i+400]

        x_in = x.unsqueeze(0).cuda()
        y_np = y[:, :, 0].cpu().numpy()

        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(x_in)

        pred_np = pred[0, :, :, 0].float().cpu().numpy()

        rel_l2 = (
            np.linalg.norm(pred_np - y_np) /
            np.linalg.norm(y_np)
        )

        vmin = min(y_np.min(), pred_np.min())
        vmax = max(y_np.max(), pred_np.max())

        # Ground Truth
        axes[i,0].imshow(y_np, vmin=vmin, vmax=vmax, cmap="RdBu_r")
        axes[i,0].set_title(f"Ground Truth #{i}", fontsize=11)
        axes[i,0].axis("off")

        # Prediction
        im = axes[i,1].imshow(pred_np, vmin=vmin, vmax=vmax, cmap="RdBu_r")
        axes[i,1].set_title(f"Predição #{i}", fontsize=11)
        axes[i,1].text(
            0.5, -0.08,
            f"Relative L2 = {rel_l2:.4f}",
            transform=axes[i,1].transAxes,
            ha="center",
            fontsize=10
        )
        axes[i,1].axis("off")

        # Error
        err_np = np.abs(pred_np - y_np)
        im_err = axes[i,2].imshow(err_np, cmap="hot")
        axes[i,2].set_title(f"Erro absoluto #{i}", fontsize=11)
        axes[i,2].axis("off")

        fig.colorbar(im, ax=[axes[i,0], axes[i,1]], fraction=0.046, pad=0.02)
        fig.colorbar(im_err, ax=axes[i,2], fraction=0.046, pad=0.02)

plt.savefig("inference_examples.png", dpi=200)
plt.show()